<a href="https://colab.research.google.com/github/mobarakol/tutorial_notebooks/blob/main/WSIBench.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#WSI-Bench Dataset

https://arxiv.org/pdf/2412.02141

Huggingface: https://huggingface.co/datasets/Lucas-yuc/WSI-Bench

In [12]:
from datasets import load_dataset
import torch

train_ds = load_dataset("Lucas-yuc/WSI-Bench", split="train")

sample = train_ds[0]
print(sample["id"])
print(sample["image"])
print(sample["conversations"])

Morphology_choice_TCGA-AO-A12A-01Z-00-DX1.4E9609A7-9AAD-40A8-8344-8369DF998006-3
TCGA-BRCA/TCGA-AO-A12A-01Z-00-DX1.4E9609A7-9AAD-40A8-8344-8369DF998006.pt
[{'from': 'human', 'value': '<image>\nWhat are the notable features of the cellular morphology in this slide? A) There is minimal variability in nuclear size, with a low rate of cell division. B) Nuclei are uniform in appearance, showing no signs of active division. C) Moderate variability in nuclear size and shape is observed, with a moderate rate of mitotic activity. D) Nuclei appear extremely pleomorphic, with a very high rate of mitotic activity.'}, {'from': 'gpt', 'value': 'C'}]


##Download all features of the images

As author indicated here: https://github.com/XinhengLyu/WSI-LLaVA/issues/2

These features were extracted using Prov-GigaPath and are used as the inputs for our WSI-LLaVA model. The files are available at: https://huggingface.co/datasets/Lucas-yuc/WSIBench-pt/tree/main.

In [7]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="Lucas-yuc/WSIBench-pt",
    repo_type="dataset",
    local_dir="/content/WSIBench-pt"
)

Fetching 528 files:   0%|          | 0/528 [00:00<?, ?it/s]

'/content/WSIBench-pt'

In [15]:
from pathlib import Path

root = Path("/content/WSIBench-pt")

num_files = len(list(root.rglob("*.pt")))

print(f"Number of .pt files: {num_files}")

Number of .pt files: 527


Download only BRCA

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="Lucas-yuc/WSIBench-pt",
    repo_type="dataset",
    local_dir="/content/WSIBench-pt",
    allow_patterns=["TCGA-BRCA/*"]
)

###Datalaoder

It seems they only released 527 .pt files out of 9850. Just read the QAs of the available .pt file

In [17]:
import torch
from torch.utils.data import Dataset
from datasets import load_dataset
from pathlib import Path


class WSIBenchDataset(Dataset):
    def __init__(self, feature_root):
        self.feature_root = Path(feature_root)

        ds = load_dataset(
            "Lucas-yuc/WSI-Bench",
            split="train"
        )

        print(f"Original QA pairs: {len(ds)}")

        valid_indices = []

        for i, sample in enumerate(ds):
            feature_path = self.feature_root / sample["image"]

            if feature_path.exists():
                valid_indices.append(i)

        self.ds = ds.select(valid_indices)

        print(f"Available QA pairs: {len(self.ds)}")

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        sample = self.ds[idx]

        feature_path = self.feature_root / sample["image"]

        image_feat = torch.load(
            feature_path,
            map_location="cpu"
        )

        question = sample["conversations"][0]["value"].replace(
            "<image>\n", ""
        )

        answer = sample["conversations"][1]["value"]

        return {
            "image": image_feat,
            "question": question,
            "answer": answer,
        }


dataset = WSIBenchDataset(
    feature_root="/content/WSIBench-pt"
)

print(len(dataset))

sample = dataset[0]
print(sample["question"])
print(sample["answer"])
print(sample["image"].shape)

Original QA pairs: 177173
Available QA pairs: 6717
6717
Based on your examination of the slide, what is the histological classification of the tumor?
The histological classification of the tumor is adenocarcinoma. This classification is informed by the presence of glandular structures with moderately preserved architecture and cellular atypia, coupled with nuclear pleomorphism. The associated intestinal metaplasia and dysplasia further support this classification.
torch.Size([576, 1024])


In [19]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="Lucas-yuc/png",
    repo_type="dataset",
    local_dir="/content/WSIBench-png",
    allow_patterns=["TCGA-BRCA/*"]
)

Fetching ... files: 0it [00:00, ?it/s]

GatedRepoError: 403 Client Error. (Request ID: Root=1-6a2b3f9f-3a4001c4334c0aca46d8c60c;cfc3682a-1ded-45c8-bbf8-f82fa8521b83)

Cannot access gated repo for url https://huggingface.co/datasets/Lucas-yuc/png/resolve/77ad7e858531e5a2c62cd9597473c0de9b424dae/TCGA-BRCA/TCGA-3C-AALI-01Z-00-DX1.F6E9A5DF-D8FB-45CF-B4BD-C6B76294C291.svs_original.png.
Your request to access dataset Lucas-yuc/png is awaiting a review from the repo authors.